### Analysis Overview

- Compares **cell type contributions** between **PreEclapmsia** and **SeverePreEclampsia** cohorts for each **deconvolution tool**.  
- Uses **Mann–Whitney U tests** to assess cohort differences within each tool.  
- Produces **boxplots with jittered points** showing PreEclapmsia and SeverePreEclampsia distributions side by side.  
- Adds **significance brackets and p-value stars** above each pair (based on raw p-values).  
- Saves both **plots (PNG/PDF)** and **statistical summaries (CSV)** for each tissue.


In [1]:
import pandas as pd
import os

# === CONFIGURATION ===
dir_path = 'TSP-BDa_Outer_300_1500_10-Moufarrej_Normotensive/'
files = [
    'CIBERSORTx_Results_Merged.txt',
    'NNLS_GSE192902_Normotensive_preQC_filtered_counts_UniquePatient_SpecificSample.txt',
    'nuSVR_Counts_TSP-BDa_Outer_300_1500_10-Moufarrej_Normotensive.txt',
    'QP_GSE192902_Normotensive_preQC_filtered_counts_UniquePatient_SpecificSample_composition.txt',
    'TSP-BDa_Inner_100each_seed42_filtered_GSE192902_Normotensive_preQC_filtered_counts_UniquePatient_SpecificSample_BayesPrism_renamed.txt',
    'TSP-HBA_Inner_100each_seed42-ReDeconv_Top1500_ReDeconv_results.tsv',
    'TSP-HBA_Inner_100each_seed42_Moufarrej_Normotensive_MuSiC.txt'
]

method_map = ['CIBERSORTx', 'MuSiC', 'QP', 'NNLS', 'BayesPrism', 'nuSVR', 'ReDeconv']

def normalize_to_100(df):
    numeric_cols = df.select_dtypes(include='number').columns
    df[numeric_cols] = df[numeric_cols].div(df[numeric_cols].sum(axis=1), axis=0) * 100
    return df

def clean_dataframe(df, file):
    if (
        file.endswith('BayesPrism_renamed.txt') or
        file.endswith('MuSiC.txt') or
        file.startswith('CIBERSORTx_Results') or
        file.endswith('ReDeconv_results.tsv')
    ):
        if file.startswith('CIBERSORTx_Results'):
            df = df.drop(columns=['P-value', 'Correlation', 'RMSE'], errors='ignore')
        df = normalize_to_100(df)

    elif file.startswith('QP') or file.startswith('NNLS'):
        df = df.drop(columns=[
            'RMSE-Composition', 'r-Composition',
            'RMSE-PredictedCounts', 'r-PredictedCounts'
        ], errors='ignore')

    return df.round(5)

def detect_method(filename):
    return next((m for m in method_map if m in filename), None)

# === FIRST PASS: COLLECT ALL CELL TYPES ===
all_celltypes = set()
temp_dfs = []
for file in files:
    file_path = os.path.join(dir_path, file)
    df = pd.read_csv(file_path, sep='\t', index_col=0)
    df = clean_dataframe(df, file)
    all_celltypes.update(df.columns)  # keep full set
    temp_dfs.append((file, df))

# Remove metadata columns from all_celltypes
all_celltypes = {c for c in all_celltypes if c not in ['Method', 'Sample']}

# === SECOND PASS: ENSURE CONSISTENT COLUMNS ===
all_results = []
for file, df in temp_dfs:
    # reindex to ensure all cell types present
    df = df.reindex(columns=list(all_celltypes))  # fill_value=None (default)

    method = detect_method(file)
    df['Method'] = method
    df['Sample'] = df.index
    all_results.append(df.reset_index(drop=True))

# === MERGE AND SAVE ===
merged_df = pd.concat(all_results, ignore_index=True)
output_file = os.path.join(dir_path, 'merged_normalised_results_Moufarrej_Normotensive.txt')
merged_df.to_csv(output_file, sep='\t', index=False)
print(f"Merged results saved to: {output_file}")


Merged results saved to: TSP-BDa_Outer_300_1500_10-Moufarrej_Normotensive/merged_normalised_results_Moufarrej_Normotensive.txt


In [3]:
import pandas as pd
import os

# === CONFIGURATION ===
dir_path = 'TSP-BDa_Outer_300_1500_10-Moufarrej_SeverePre-eclampsia'
files = [
    'CIBERSORTx_Results.txt',
    'NNLS_GSE192902_SeverePre-eclampsia_preQC_filtered_counts_UniquePatient_SpecificSample.txt',
    'nuSVR_Counts_TSP-BDa_Outer_300_1500_10-Moufarrej_SeverePre-eclampsia.txt',
    'QP_GSE192902_SeverePre-eclampsia_preQC_filtered_counts_UniquePatient_SpecificSample_composition.txt',
    'TSP-BDa_Inner_100each_seed42_filtered_GSE192902_SeverePre-eclampsia_preQC_filtered_counts_UniquePatient_SpecificSample_BayesPrism_renamed.txt',
    'TSP-HBA_Inner_100each_seed42-ReDeconv_Top1500_ReDeconv_results.tsv',
    'TSP-HBA_Inner_100each_seed42_Moufarrej_SeverePre-eclampsia_MuSiC.txt'
]

method_map = ['CIBERSORTx', 'MuSiC', 'QP', 'NNLS', 'BayesPrism', 'nuSVR', 'ReDeconv']

def normalize_to_100(df):
    numeric_cols = df.select_dtypes(include='number').columns
    df[numeric_cols] = df[numeric_cols].div(df[numeric_cols].sum(axis=1), axis=0) * 100
    return df

def clean_dataframe(df, file):
    if (
        file.endswith('BayesPrism_renamed.txt') or
        file.endswith('MuSiC.txt') or
        file.startswith('CIBERSORTx_Results') or
        file.endswith('ReDeconv_results.tsv')
    ):
        if file.startswith('CIBERSORTx_Results'):
            df = df.drop(columns=['P-value', 'Correlation', 'RMSE'], errors='ignore')
        df = normalize_to_100(df)

    elif file.startswith('QP') or file.startswith('NNLS'):
        df = df.drop(columns=[
            'RMSE-Composition', 'r-Composition',
            'RMSE-PredictedCounts', 'r-PredictedCounts'
        ], errors='ignore')

    return df.round(5)

def detect_method(filename):
    return next((m for m in method_map if m in filename), None)

# === FIRST PASS: COLLECT ALL CELL TYPES ===
all_celltypes = set()
temp_dfs = []
for file in files:
    file_path = os.path.join(dir_path, file)
    df = pd.read_csv(file_path, sep='\t', index_col=0)
    df = clean_dataframe(df, file)
    all_celltypes.update(df.columns)  # keep full set
    temp_dfs.append((file, df))

# Remove metadata columns from all_celltypes
all_celltypes = {c for c in all_celltypes if c not in ['Method', 'Sample']}

# === SECOND PASS: ENSURE CONSISTENT COLUMNS ===
all_results = []
for file, df in temp_dfs:
    # reindex to ensure all cell types present
    df = df.reindex(columns=list(all_celltypes))  # fill_value=None (default)

    method = detect_method(file)
    df['Method'] = method
    df['Sample'] = df.index
    all_results.append(df.reset_index(drop=True))

# === MERGE AND SAVE ===
merged_df = pd.concat(all_results, ignore_index=True)
output_file = os.path.join(dir_path, 'merged_normalised_results_Moufarrej_SeverePre-Eclampsia.txt')
merged_df.to_csv(output_file, sep='\t', index=False)
print(f"Merged results saved to: {output_file}")


Merged results saved to: TSP-BDa_Outer_300_1500_10-Moufarrej_SeverePre-eclampsia/merged_normalised_results_Moufarrej_SeverePre-Eclampsia.txt


In [ ]:
import pandas as pd
import os

# === CONFIGURATION ===
dir_path = 'TSP-BDa_Outer_300_1500_10-Moufarrej_Pre-eclampsia'
files = [
    'CIBERSORTx_Results.txt',
    'NNLS_GSE192902_Pre-eclampsia_preQC_filtered_counts_UniquePatient_SpecificSample.txt',
    'nuSVR_Counts_TSP-BDa_Outer_300_1500_10-Moufarrej_Pre-eclampsia.txt',
    'QP_GSE192902_Pre-eclampsia_preQC_filtered_counts_UniquePatient_SpecificSample_composition.txt',
    'TSP-BDa_Inner_100each_seed42_filtered_GSE192902_Pre-eclampsia_preQC_filtered_counts_UniquePatient_SpecificSample_BayesPrism_renamed.txt',
    'TSP-HBA_Inner_100each_seed42-ReDeconv_Top1500_ReDeconv_results.tsv',
    'TSP-HBA_Inner_100each_seed42_Moufarrej_Pre-eclampsia_MuSiC.txt'
]

method_map = ['CIBERSORTx', 'MuSiC', 'QP', 'NNLS', 'BayesPrism', 'nuSVR', 'ReDeconv']

def normalize_to_100(df):
    numeric_cols = df.select_dtypes(include='number').columns
    df[numeric_cols] = df[numeric_cols].div(df[numeric_cols].sum(axis=1), axis=0) * 100
    return df

def clean_dataframe(df, file):
    if (
        file.endswith('BayesPrism_renamed.txt') or
        file.endswith('MuSiC.txt') or
        file.startswith('CIBERSORTx_Results') or
        file.endswith('ReDeconv_results.tsv')
    ):
        if file.startswith('CIBERSORTx_Results'):
            df = df.drop(columns=['P-value', 'Correlation', 'RMSE'], errors='ignore')
        df = normalize_to_100(df)

    elif file.startswith('QP') or file.startswith('NNLS'):
        df = df.drop(columns=[
            'RMSE-Composition', 'r-Composition',
            'RMSE-PredictedCounts', 'r-PredictedCounts'
        ], errors='ignore')

    return df.round(5)

def detect_method(filename):
    return next((m for m in method_map if m in filename), None)

# === FIRST PASS: COLLECT ALL CELL TYPES ===
all_celltypes = set()
temp_dfs = []
for file in files:
    file_path = os.path.join(dir_path, file)
    df = pd.read_csv(file_path, sep='\t', index_col=0)
    df = clean_dataframe(df, file)
    all_celltypes.update(df.columns)  # keep full set
    temp_dfs.append((file, df))

# Remove metadata columns from all_celltypes
all_celltypes = {c for c in all_celltypes if c not in ['Method', 'Sample']}

# === SECOND PASS: ENSURE CONSISTENT COLUMNS ===
all_results = []
for file, df in temp_dfs:
    # reindex to ensure all cell types present
    df = df.reindex(columns=list(all_celltypes))  # fill_value=None (default)

    method = detect_method(file)
    df['Method'] = method
    df['Sample'] = df.index
    all_results.append(df.reset_index(drop=True))

# === MERGE AND SAVE ===
merged_df = pd.concat(all_results, ignore_index=True)
output_file = os.path.join(dir_path, 'merged_normalised_results_Moufarrej_Pre-Eclampsia.txt')
merged_df.to_csv(output_file, sep='\t', index=False)
print(f"Merged results saved to: {output_file}")

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests
import matplotlib as mpl


# ---------- Helper function ----------
def pval_to_star(p):
    if p < 0.0001: return "****"
    elif p < 0.001: return "***"
    elif p < 0.01: return "**"
    elif p < 0.05: return "*"
    else: return "ns"

# ---------- Load data ----------
files = {
    "PreEclampsia": "TSP-BDa_Outer_300_1500_10-Moufarrej_Pre-eclampsia/merged_normalised_results_Moufarrej_Pre-Eclampsia.txt",
    "SeverePreEclampsia": "TSP-BDa_Outer_300_1500_10-Moufarrej_SeverePre-eclampsia/merged_normalised_results_Moufarrej_SeverePre-Eclampsia.txt",
    "Normotensive": "TSP-BDa_Outer_300_1500_10-Moufarrej_Normotensive/merged_normalised_results_Moufarrej_Normotensive.txt"
}

dfs = []
for cohort, path in files.items():
    df = pd.read_csv(path, sep="\t")
    df["Cohort"] = cohort
    dfs.append(df)

merged = pd.concat(dfs, ignore_index=True)
print(f"Loaded {merged.shape[0]} samples across {merged['Method'].nunique()} methods.")

# ---------- Define groups ----------
merged["Group"] = merged["Cohort"].apply(
    lambda c: "Diseased" if c in ["PreEclampsia", "SeverePreEclampsia"] else "Healthy"
)

# ---------- Reshape ----------
long_df = merged.melt(
    id_vars=["Sample", "Method", "Cohort", "Group"],
    var_name="CellType",
    value_name="Contribution"
)
long_df["Contribution"] = pd.to_numeric(long_df["Contribution"], errors="coerce")
long_df = long_df.dropna(subset=["Contribution"])

# ---------- Statistical testing ----------
results = []
for (method, celltype), df_sub in long_df.groupby(["Method", "CellType"]):
    group_data = {g: vals["Contribution"].values for g, vals in df_sub.groupby("Group")}
    if "Diseased" in group_data and "Healthy" in group_data:
        dis_vals = group_data["Diseased"]
        ctrl_vals = group_data["Healthy"]

        if len(dis_vals) > 0 and len(ctrl_vals) > 0:
            try:
                # Compute medians and Mann–Whitney U test
                median_dis = np.median(dis_vals)
                median_ctrl = np.median(ctrl_vals)
                delta = median_dis - median_ctrl
                stat, pval = mannwhitneyu(dis_vals, ctrl_vals, alternative="two-sided")

                results.append({
                    "Method": method,
                    "CellType": celltype,
                    "pval": pval,
                    "Significant": pval < 0.05,
                    "Stars": pval_to_star(pval),
                    "Median_Diseased": median_dis,
                    "Median_Healthy": median_ctrl,
                    "Delta_Diseased_minus_Healthy": delta,
                    "Direction": "Diseased > Healthy" if delta > 0 else "Healthy > Diseased"
                })
            except Exception as e:
                print(f"Skipping {method} - {celltype}: {e}")

# ---------- Collect results ----------
results_df = pd.DataFrame(results)

# ---------- Apply Benjamini–Hochberg (FDR) correction (WITHIN TOOL / Method) ----------
results_df["pval_BH"] = np.nan

for method, idx in results_df.groupby("Method").groups.items():
    valid = results_df.loc[idx, "pval"].dropna()
    if len(valid) == 0:
        continue
    _, p_corr, _, _ = multipletests(valid, method="fdr_bh")
    results_df.loc[valid.index, "pval_BH"] = p_corr

results_df["Significant_BH"] = results_df["pval_BH"] < 0.05
results_df["Stars_BH"] = results_df["pval_BH"].apply(pval_to_star)

# ---------- Print FDR-significant results ----------
sig_bh = results_df[results_df["Significant_BH"]].sort_values("pval_BH")
print("\n=== Statistically Significant Differences (FDR < 0.05, BH-corrected) ===")
if sig_bh.empty:
    print("No significant FDR-corrected differences found between Diseased and Healthy cohorts.")
else:
    for _, row in sig_bh.iterrows():
        arrow = "↑" if row["Delta_Diseased_minus_Healthy"] > 0 else "↓"
        print(f"{row['Method']:12s} | {row['CellType'][:40]:40s} | "
              f"FDR = {row['pval_BH']:.3e} {row['Stars_BH']:4s} | "
              f"{row['Direction']:9s} ({arrow}) | Δ = {row['Delta_Diseased_minus_Healthy']:.3f}")

# ---------- Print uncorrected significant results ----------
print("\n=== Statistically Significant Differences (p < 0.05, uncorrected) ===")
sig_df = results_df[results_df["Significant"]].sort_values("pval")
if sig_df.empty:
    print("No significant differences found between Diseased and Healthy cohorts.")
else:
    for _, row in sig_df.iterrows():
        arrow = "↑" if row["Delta_Diseased_minus_Healthy"] > 0 else "↓"
        print(f"{row['Method']:12s} | {row['CellType'][:40]:40s} | "
              f"p = {row['pval']:.3e} {row['Stars']:4s} | "
              f"{row['Direction']:9s} ({arrow}) | Δ = {row['Delta_Diseased_minus_Healthy']:.3f}")

# ---------- Save results ----------
results_df.to_csv("AllCellTypes_Moufarrej_Diseased_vs_Healthy_BH_WithinTool.csv", index=False)
print("\nFull statistical results saved to: AllCellTypes_Moufarrej_Diseased_vs_Healthy_BH_WithinTool.csv")


Loaded 868 samples across 7 methods.


/tmp/ipykernel_3000592/1778561251.py:31: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  merged["Group"] = merged["Cohort"].apply(



=== Statistically Significant Differences (FDR < 0.05, BH-corrected) ===
BayesPrism   | serous cell of epithelium of trachea     | FDR = 7.995e-08 **** | Healthy > Diseased (↓) | Δ = -0.001
BayesPrism   | endothelial cell of arteriole/endothelia | FDR = 3.283e-05 **** | Healthy > Diseased (↓) | Δ = -0.003
BayesPrism   | innate lymphoid cell                     | FDR = 6.167e-05 **** | Healthy > Diseased (↓) | Δ = -0.002
BayesPrism   | myometrial cell                          | FDR = 1.221e-04 ***  | Healthy > Diseased (↓) | Δ = -0.003
BayesPrism   | pancreatic acinar cell                   | FDR = 1.221e-04 ***  | Healthy > Diseased (↓) | Δ = -0.000
BayesPrism   | enteroglial cell                         | FDR = 1.394e-04 ***  | Healthy > Diseased (↓) | Δ = -0.001
BayesPrism   | tissue-resident macrophage               | FDR = 2.058e-04 ***  | Healthy > Diseased (↓) | Δ = -0.002
BayesPrism   | basophil/mast cell                       | FDR = 2.058e-04 ***  | Healthy > Diseased (↓) | Δ

In [2]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib import rcParams, patches
import matplotlib as mpl

sns.set_theme(style="white", context="paper")  # context scales fonts nicely

mpl.rcParams.update({
    # Fonts (Illustrator-friendly)
    "font.family": "DejaVu Sans",
    "font.sans-serif": ["DejaVu Sans"],

    # Keep text as text in vector outputs
    "svg.fonttype": "none",
    "pdf.fonttype": 42,
    "ps.fonttype": 42,

    # Sizes
    "font.size": 9,
    "axes.labelsize": 12,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "axes.titlesize": 13,

    # Lines
    "axes.linewidth": 0.8,
    "hatch.linewidth": 1.2,

    # Save
    "savefig.bbox": "tight",
    "savefig.transparent": True,
})

def fig_for_heatmap(data, cell_in=0.30,
                    left_in=1.1, right_in=0.9,
                    bottom_in=1.0, top_in=0.3):
    """
    Figure size is derived from the heatmap matrix shape.
    Cell size is fixed (cell_in), margins are fixed (inches).
    """
    nrows, ncols = data.shape
    fig_w = ncols * cell_in + left_in + right_in
    fig_h = nrows * cell_in + top_in + bottom_in

    fig, ax = plt.subplots(figsize=(fig_w, fig_h))
    fig.subplots_adjust(
        left=left_in / fig_w,
        right=1 - right_in / fig_w,
        bottom=bottom_in / fig_h,
        top=1 - top_in / fig_h
    )
    return fig, ax

# ---------- Cell type mapping (expanded “variability” set) ----------
celltype_map = {
    # Multi-tool diseased↑ anchors
    "erythrocyte/platelet": "Erythrocyte/Platelet",
    "macrophage": "Macrophage",
    "common myeloid progenitor/hematopoietic": "Myeloid progenitor (CMP/HSPC)",
    "intestinal tuft cell/large intestine gob": "Intestinal tuft cell",
    "type i pneumocyte": "Type I pneumocyte",
    "neurons": "Neurons",

    # PE-relevant / blood-immune contrast (often Healthy > Diseased)
    "classical monocyte/intermediate monocyte": "Monocyte (classical/intermediate)",
    "plasma cell": "Plasma cell",
    "activated cd4-positive, alpha-beta t cel": "Activated CD4 T cell",
    "innate lymphoid cell": "Innate lymphoid cell",
    "basophil/mast cell": "Basophil/Mast cell",
    "leukocyte": "Leukocyte (broad)",

    # Vascular contrast (often Healthy > Diseased)
    "endothelial cell of arteriole/endothelia": "Endothelial cell (arteriole/capillary)",
    "capillary endothelial cell/endothelial c": "Endothelial cell (arteriole/capillary)",
    "mural cell/pericyte": "Mural cell/Pericyte",

    # Pregnancy-themed contrast
    "myometrial cell": "Myometrial cell",
    "epithelial cell of uterus": "Uterus epithelial cell",

    # “Unexpected epithelial” (nice for demonstrating breadth/variability)
    "serous cell of epithelium of trachea": "Tracheal serous cell",
    "mucus secreting cell/respiratory goblet": "Respiratory goblet cell",

    # Optional single-tool contrast
    "neutrophil": "Neutrophil",
    "kidney epithelial cell": "Kidney epithelial cell",
}

# ---------- Load precomputed statistical results ----------
results_df = pd.read_csv("AllCellTypes_Moufarrej_Diseased_vs_Healthy_BH_WithinTool.csv")

# ---------- Column name handling (robust) ----------
possible_delta_cols = [
    "Delta_Diseased_minus_Healthy",
    "Delta_AD_minus_NCI",
    "Delta",
    "Delta_median",
    "Delta_median_contribution",
]
delta_col = next((c for c in possible_delta_cols if c in results_df.columns), None)
if delta_col is None:
    raise ValueError(
        "Could not find a delta column. Expected one of: "
        + ", ".join(possible_delta_cols)
        + f"\nAvailable columns: {list(results_df.columns)}"
    )

possible_star_cols = ["Stars_BH", "Stars", "Signif", "BH_Stars"]
star_col = next((c for c in possible_star_cols if c in results_df.columns), None)

# ---------- Filter to selected cell types ----------
results_df = results_df[results_df["CellType"].isin(celltype_map.keys())].copy()
results_df["CellTypeGroup"] = results_df["CellType"].map(celltype_map)

# Clean stars if present
if star_col is not None:
    results_df.loc[results_df[star_col].astype(str).str.lower() == "ns", star_col] = ""
else:
    star_col = "Stars_BH"
    results_df[star_col] = ""

# ---------- Prepare heatmap data ----------
heatmap_data = results_df.pivot(index="CellTypeGroup", columns="Method", values=delta_col)
annot_data   = results_df.pivot(index="CellTypeGroup", columns="Method", values=star_col)

# ---------- Order rows and columns ----------
tool_order = ["BayesPrism", "MuSiC", "nuSVR", "CIBERSORTx", "NNLS", "QP", "ReDeconv"]

cell_order = [
    # Blood / immune anchors
    "Erythrocyte/Platelet",
    "Leukocyte (broad)",
    "Monocyte (classical/intermediate)",
    "Macrophage",
    "Neutrophil",
    "Plasma cell",
    "Activated CD4 T cell",
    "Innate lymphoid cell",
    "Basophil/Mast cell",

    # Vascular
    "Endothelial cell (arteriole/capillary)",
    "Mural cell/Pericyte",

    # Core multi-tool epithelial/neuro variability
    "Kidney epithelial cell",
    "Type I pneumocyte",
    "Respiratory goblet cell",
    "Tracheal serous cell",
    "Intestinal tuft cell",
    "Neurons",

    # Pregnancy-themed
    "Myometrial cell",
    "Uterus epithelial cell",
]

# Reindex safely (keeps only those present)
heatmap_data = heatmap_data.reindex([c for c in cell_order if c in heatmap_data.index])
annot_data   = annot_data.reindex([c for c in cell_order if c in annot_data.index])

# Order tools that are present
tools_present = [t for t in tool_order if t in heatmap_data.columns]
heatmap_data = heatmap_data[tools_present]
annot_data   = annot_data[tools_present]

mask = heatmap_data.isna()

# ---------- Transpose for plotting ----------
heatmap_plot = heatmap_data.T
annot_plot   = annot_data.T
mask_plot    = mask.T

# ---------- Plot (fixed cell size, content-driven figure size) ----------
cmap = plt.cm.get_cmap("RdBu_r").copy()
cmap.set_bad(color="lightgrey")
fig, ax = fig_for_heatmap(
    heatmap_plot,
    cell_in=0.30,   # cell square size (inches)
    left_in=1.1,    # space for y labels
    right_in=0.9,   # space for colorbar
    bottom_in=1.0,  # space for rotated x labels
    top_in=0.3
)

ax = sns.heatmap(
    heatmap_plot,
    annot=annot_plot,
    fmt="",
    cmap=cmap,
    center=0,
    linewidths=0.7,
    linecolor="white",
    mask=mask_plot,
    square=True,  # <-- keeps the cells square
    annot_kws={"fontsize": 9, "fontweight": "bold"},
    ax=ax
)

# Hatched overlay for missing
for y in range(heatmap_plot.shape[0]):
    for x in range(heatmap_plot.shape[1]):
        if mask_plot.iloc[y, x]:
            ax.add_patch(
                patches.Rectangle(
                    (x, y), 1, 1,
                    facecolor="none",
                    edgecolor="white",
                    hatch="////",
                    linewidth=0.0
                )
            )

#ax.set_title("Difference in cell-type proportions (Diseased − Healthy)", pad=10, fontweight="bold")
ax.set_xlabel("Cell Type", fontsize=9, labelpad=6)
ax.set_ylabel("Deconvolution Tool", fontsize=9, labelpad=6)

ax.tick_params(axis="x", labelsize=8)
ax.tick_params(axis="y", labelsize=8)
plt.setp(ax.get_xticklabels(), rotation=45, ha="right", rotation_mode="anchor")

# ---------- Stars formatting ----------
star_set = {"*", "**", "***", "****"}
for t in ax.texts:
    if t.get_text() in star_set:
        t.set_fontsize(7)
        t.set_fontweight("bold")
        t.set_ha("center")
        t.set_va("center")

# ---- Colorbar formatting ----
cbar = ax.collections[0].colorbar
cbar.ax.tick_params(labelsize=7)
cbar.set_label(
    "Δ (Preeclampsia − Normotensive) median contribution (%)",
    fontsize=8,
    rotation=270,
    labelpad=12
)

sns.despine(left=True, bottom=True)

# ---------- Save  ----------
fig.savefig("Heatmap_Preeclampsia_vs_Normotensive_Moufarrej_BH_corrected_Transposed_BH_WithinTool.svg", format="svg", bbox_inches="tight")
plt.close(fig)

/tmp/ipykernel_3000592/3964641128.py:177: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  cmap = plt.cm.get_cmap("RdBu_r").copy()


In [3]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib as mpl

mpl.rcParams["svg.fonttype"] = "none"
mpl.rcParams["font.family"] = "DejaVu Sans"
mpl.rcParams["font.sans-serif"] = ["DejaVu Sans"]

def add_bracket(ax, x1, x2, y, text, height):
    if text in ["", None] or pd.isna(text):
        return
    ax.plot([x1, x1, x2, x2], [y, y+height, y+height, y], lw=1.2, c="black")
    ax.text((x1+x2)/2, y+height, text, ha="center", va="bottom",
            fontsize=8, fontweight="bold", color="black")

# ---------- Settings ----------
tool_order = ["BayesPrism", "MuSiC", "nuSVR", "CIBERSORTx", "NNLS", "QP", "ReDeconv"]
palette = {"Healthy": "#4C72B0", "Diseased": "#DD1C1A"}
hue_offsets = {"Healthy": -0.20, "Diseased": 0.20}

# cell types to plot (raw names -> pretty names)
targets = {
    "erythrocyte/platelet": "Erythrocyte/Platelet",
    "macrophage": "Macrophage",
}

ylim_map = {
    "Macrophage": (0, 10),
    "Erythrocyte/Platelet": (0, 100),
}

sns.set_theme(style="white")
sns.set_context("paper", font_scale=1.0)

for cell_raw, cell_pretty in targets.items():

    # --- subset raw data for plotting ---
    df_ct = long_df[long_df["CellType"].eq(cell_raw)].copy()
    if df_ct.empty:
        print(f"Skipping {cell_pretty}: no data for CellType='{cell_raw}'.")
        continue

    # keep only tools that exist for this celltype (avoids empty MuSiC-like columns)
    tools_present = [t for t in tool_order if t in df_ct["Method"].unique()]

    df_ct["Method"] = pd.Categorical(df_ct["Method"], categories=tools_present, ordered=True)
    df_ct["Group"]  = pd.Categorical(df_ct["Group"], categories=["Healthy", "Diseased"], ordered=True)

    # --- stars per tool from BH-within-tool results (keep 'ns') ---
    res_ct = results_df[results_df["CellType"].eq(cell_raw)].copy()
    res_ct["Stars_plot"] = res_ct["Stars_BH"].fillna("ns")

    stars_map = (
        res_ct[["Method", "Stars_plot"]]
        .dropna(subset=["Method"])
        .set_index("Method")["Stars_plot"]
        .to_dict()
    )

    # Ensure EVERY plotted tool gets a label (default to ns)
    for tool in tools_present:
        if tool not in stars_map or stars_map[tool] in ["", None] or pd.isna(stars_map[tool]):
            stars_map[tool] = "ns"


    fig, ax = plt.subplots(figsize=(5, 4))

    sns.boxplot(
        data=df_ct, x="Method", y="Contribution", hue="Group",
        palette=palette, dodge=True, showfliers=False, ax=ax
    )
    sns.stripplot(
        data=df_ct, x="Method", y="Contribution", hue="Group",
        palette=palette, dodge=True, jitter=True, alpha=0.6,
        linewidth=0.5, size=4, ax=ax
    )

    # one legend
    handles, labels = ax.get_legend_handles_labels()
    ax.legend(handles[:2], labels[:2], title="Group", frameon=True)

    # ----- Set y-limits per cell type -----
    if cell_pretty in ylim_map:
        ax.set_ylim(*ylim_map[cell_pretty])

    # bracket placement inside axis (works with or without fixed ylim)
    y0, y1 = ax.get_ylim()
    yr = max(y1 - y0, 1e-6)
    base_y = y0 + 0.90 * yr
    height = 0.3

    # tool -> x
    xticks = ax.get_xticks()
    xticklabels = [t.get_text() for t in ax.get_xticklabels()]
    tool_to_x = dict(zip(xticklabels, xticks))

    for tool in tools_present:
        star = stars_map.get(tool, "ns")  # if missing, show ns
        if tool not in tool_to_x:
            continue
        x = tool_to_x[tool]
        add_bracket(ax,
                    x + hue_offsets["Healthy"],
                    x + hue_offsets["Diseased"],
                    base_y, star, height)

    ax.set_xlabel("Deconvolution Tool", fontsize=10)
    ax.set_ylabel(f"{cell_pretty} contribution (%)", fontsize=10)
    ax.grid(True, linestyle="--", linewidth=0.5, alpha=0.7)
    plt.setp(ax.get_xticklabels(), rotation=30, ha="right")
    sns.despine()
    plt.tight_layout()

    out = f"Moufarrej_Diseased_vs_Healthy_{cell_pretty.replace('/','-')}_boxstrip_BHWithinTool.svg"
    plt.savefig(out, format="svg", bbox_inches="tight")
    plt.close(fig)

    print(f"Saved: {out}")


Saved: Moufarrej_Diseased_vs_Healthy_Erythrocyte-Platelet_boxstrip_BHWithinTool.svg
Saved: Moufarrej_Diseased_vs_Healthy_Macrophage_boxstrip_BHWithinTool.svg
